In [2]:
from openaq import OpenAQ
from dotenv import load_dotenv
import os
from pprint import pprint
import json
from tqdm import tqdm
import pandas as pd
from time import sleep
import logging

logging.basicConfig(filename='scrape_data.log', level=logging.ERROR, format='%(asctime)s:%(levelname)s:%(message)s')

load_dotenv(override=True)
openaq_api_key = os.getenv("OPENAQ_API_KEY")
openaq_client = OpenAQ(api_key=openaq_api_key, )

In [3]:
from datetime import datetime, timedelta

def split_period(start, end):
    start = datetime.fromisoformat(start)
    end = datetime.fromisoformat(end)
    
    year_count = abs(end.year - start.year) + 1
    n = 48 * year_count

    delta = (end - start) / n
    intervals = []
    for i in range(n):
        begin = start + i * delta
        finish = begin + delta
        intervals.append((begin.isoformat(), finish.isoformat()))
    return intervals

In [4]:
# lấy tất cả các trạm quan trắc ở Việt Nam
vn_locations = openaq_client.locations.list(limit=1000, countries_id=56)
print(len(vn_locations.results))

53


In [5]:
# lấy các trạm quan trắc ở Hà Nội
hn_locations = [loc for loc in vn_locations.results if loc.locality == "Hanoi"]

In [13]:
# import ast
# ast.literal_eval(dff.loc[0, "period"])

In [7]:
try:
    for i in range(37, len(hn_locations)):
        loc = hn_locations[i]
        loc_sensors = openaq_client.locations.sensors(loc.id)

        if loc_sensors.headers.x_ratelimit_remaining == 0:
            sleep(60)

        for loc_sensor in loc_sensors.results:
            datetime_first = loc_sensor.datetime_first
            datetime_last = loc_sensor.datetime_last
            periods = split_period(datetime_first.utc, datetime_last.utc)
            for period in tqdm(periods, desc=f"location: {loc.id} sensor: {loc_sensor.id} period: {datetime_first.utc} - {datetime_last.utc}"):
                measurements = openaq_client.measurements.list(sensors_id=loc_sensor.id, datetime_from=period[0], datetime_to=period[1], limit=1000)
                if int(measurements.meta.found) > 0:
                    df = pd.DataFrame(measurements.results)
                    df.to_csv(f"../csv_data/loc{loc.id}_sen{loc_sensor.id}.csv", index=False, encoding="utf-8", mode="a")
                
                if measurements.headers.x_ratelimit_remaining == 0:
                    sleep(60)
                sleep(0.4)

except Exception as e:
    logging.error(f"Error processing location {loc.id}, sensor {loc_sensor.id}, period {period[0]} to {period[1]}: {e}")
    log = {"location_id": loc.id, "sensor_id": loc_sensor.id, "period_start": period[0], "period_end": period[1], "error": str(e)}
    pd.DataFrame([log]).to_csv("../csv_scrape_error/scrape_error.csv", index=False, encoding="utf-8", mode="a")
except KeyboardInterrupt as ke:
    logging.error("Scraping interrupted by user.")
    print("Scraping interrupted by user.")
    log = {"location_id": loc.id, "sensor_id": loc_sensor.id, "period_start": period[0], "period_end": period[1], "error": str(ke)}
    pd.DataFrame([log]).to_csv("../csv_scrape_error/scrape_error.csv", index=False, encoding="utf-8", mode="a")

location: 4946813 sensor: 13502149 period: 2025-07-03T15:55:00Z - 2026-03-04T07:30:00Z: 100%|██████████| 96/96 [03:02<00:00,  1.90s/it]
location: 4946813 sensor: 13502152 period: 2025-07-03T15:40:00Z - 2026-03-04T07:30:00Z: 100%|██████████| 96/96 [03:08<00:00,  1.96s/it]
location: 4946813 sensor: 13502154 period: 2025-07-03T15:40:00Z - 2026-03-04T07:30:00Z: 100%|██████████| 96/96 [03:04<00:00,  1.93s/it]
location: 4946813 sensor: 13502155 period: 2025-07-03T15:40:00Z - 2026-03-04T07:30:00Z: 100%|██████████| 96/96 [03:35<00:00,  2.25s/it]
location: 4946813 sensor: 13502156 period: 2025-07-03T15:40:00Z - 2025-11-18T10:35:00Z: 100%|██████████| 48/48 [01:26<00:00,  1.79s/it]
location: 4946813 sensor: 13502164 period: 2025-07-03T15:40:00Z - 2026-03-04T07:30:00Z: 100%|██████████| 96/96 [03:04<00:00,  1.92s/it]


In [9]:
for i in range(len(hn_locations)):
    if hn_locations[i].id == 4946813:
        print(i)
        break

37


In [8]:
import winsound

winsound.PlaySound("SystemExclamation", winsound.SND_ALIAS)